# Notebook para arreglar y mejorar el juego de Glob Defenders
Esta notebook revisa la generación de oleadas, la visibilidad del ataque de Glob Cometa, el comportamiento de cierre de modales y corrige caracteres extraños en `config.js`.

## 1. Revisar y Mejorar Diversidad en las Oleadas
Analiza y actualiza la lógica de generación de oleadas para mayor variedad de enemigos.

In [ ]:
from pathlib import Path
path = Path('game.js')
text = path.read_text(encoding='utf-8')
pattern = 'function spawnEnemy(type, boss) {
  if (!type) {
    const keys = Object.keys(ENEMY_TYPES).filter(k => k !== Stupid_GoldPyce && k !== Mimic_Pyce && k !== Flower_Pyce);
    type = keys[Math.floor(Math.random() * Math.min(keys.length, Math.floor(gameState.wave/3)+1))];
    
    // Chance de reemplazo por Mimic o Flower Pyce
    const roll = Math.random();
    if (roll < 0.01) type = Mimic_Pyce;
    else if (roll < 0.11) type = Stupid_GoldPyce; // 10% aprox (0.01 a 0.11)
    else if (roll < 0.21 && gameState.wave > 5) type = Flower_Pyce; // Flower Pyce a partir de oleada 5
  }
'
replacement = '''function spawnEnemy(type, boss) {
  if (!type) {
    const wave = gameState.wave;
    const pool = ['Stupid_Pyce'];

    if (wave >= 2) {
      pool.push('Pyce2', 'Pyce2');
    }
    if (wave >= 3) {
      pool.push('Guest_Pyce', 'Symbol_Pyce');
    }
    if (wave >= 5) {
      pool.push('Noob_Pyce', 'Noob_Pyce');
    }
    if (wave >= 8) {
      pool.push('4motions_Pyce');
    }
    if (wave >= 10) {
      pool.push('SO_Pyce', 'Symbol_Pyce', 'Guest_Pyce', 'Noob_Pyce');
    }
    if (wave >= 12) {
      pool.push('SO_Pyce', '4motions_Pyce');
    }

    if (Math.random() < 0.2 && wave >= 3) pool.push('Symbol_Pyce');
    if (Math.random() < 0.15 && wave >= 6) pool.push('Noob_Pyce');
    if (Math.random() < 0.1 && wave >= 8) pool.push('4motions_Pyce');

    type = pool[Math.floor(Math.random() * pool.length)] || 'Stupid_Pyce';

    const roll = Math.random();
    if (roll < 0.01) type = 'Mimic_Pyce';
    else if (roll < 0.11) type = 'Stupid_GoldPyce';
    else if (roll < 0.21 && wave > 5) type = 'Flower_Pyce';
  }
'''
if pattern in text:
    text = text.replace(pattern, replacement)
    path.write_text(text, encoding='utf-8')
    print('Replaced spawnEnemy logic.')
else:
    print('spawnEnemy pattern not found.')

## 2. Corregir Visualización de Ataques de Glob Cometa
Asegura que el proyectil `blue_comet` tenga estilo visible y usable.

In [ ]:
path = Path('styles.css')
text = path.read_text(encoding='utf-8')
needle = '.projectile.green { background: #2ecc71; width: 10px; height: 10px; box-shadow: 0 0 5px #2ecc71; }
'
insert = '''.projectile.blue_comet { background: linear-gradient(135deg, #5fc3ff, #0d6fff); width: 14px; height: 14px; box-shadow: 0 0 12px rgba(95, 195, 255, 0.8); border-radius: 50%; }
'''
if insert in text:
    print('blue_comet style already present.')
else:
    if needle in text:
        text = text.replace(needle, needle + insert)
        path.write_text(text, encoding='utf-8')
        print('Inserted blue_comet style.')
    else:
        print('Could not find insertion point for blue_comet style.')

## 3. Mantener Progreso al Salir de Tienda o Pase
Asegura que el cierre de modales con la 'X' no regrese a selección de modo después de haber elegido un modo.

In [ ]:
path = Path('game.js')
text = path.read_text(encoding='utf-8')
changed = False
if 'modeConfirmed: false,' not in text:
    text = text.replace('mode: normal,
  corrupt: false,', 'mode: normal,
  modeConfirmed: false,
  corrupt: false,')
    changed = True
if 'gameState.modeConfirmed = true;' not in text and 'gameState.mode = mode;' in text:
    text = text.replace('gameState.mode = mode;
  const limits = { facil: 10, normal: 15, dificil: 25, extremo: 40, infinito: 999, corrupto: 45 };', 'gameState.mode = mode;
  gameState.modeConfirmed = true;
  const limits = { facil: 10, normal: 15, dificil: 25, extremo: 40, infinito: 999, corrupto: 45 };')
    changed = True
smart = 'function smartClose(modalId) {
  closeModal(modalId);
  const modeScreen = document.getElementById(mode-selection);
  if (document.getElementById(login-screen).style.display === none) {
    modeScreen.style.display = flex;
  }
}'
replacement = 'function smartClose(modalId) {
  closeModal(modalId);
  const modeScreen = document.getElementById(mode-selection);
  if (!gameState.modeConfirmed && document.getElementById(login-screen).style.display === none) {
    modeScreen.style.display = flex;
  }
}'
if smart in text:
    text = text.replace(smart, replacement)
    changed = True
if changed:
    path.write_text(text, encoding='utf-8')
    print('Updated game state and smartClose logic.')
else:
    print('No changes needed in game.js for progress preservation.')

## 4. Limpiar Caracteres Extraños en Configuración
Ejecuta un script para corregir caracteres mojibake y normalizar las traducciones en `config.js`.

In [ ]:
path = Path('config.js')
text = path.read_text(encoding='utf-8')
replacements = {
        'Ã¡': 'á', 'Ã©': 'é', 'Ã­': 'í', 'Ã³': 'ó', 'Ãº': 'ú', 'Ã±': 'ñ', 'Ã¼': 'ü',
        'Â¡': '¡', 'Â¿': '¿', 'âŒ': '❌', 'âœ…': '✓', 'âœ¨': '⭐', 'âš™': '⚙️',
        'âš¡': '⏱️', 'âš”': '⚔️', 'ðŸ’€': '💀', 'ðŸ’¸': '💸', 'ðŸŽ‰': '✨',
        'ðŸ”“': '🌀', 'ðŸ‘‘': '👑', 'ðŸ’µ': '🛍️', 'ðŸŒŠ': '🌊', 'ðŸ’”': '⚠️',
        'ðŸ†': '🏆', 'ðŸ‘€': '👉', 'ðŸ¤”': '🤨', 'ðŸš¨': '🧨', 'Â·': '·',
        'Â°': '°', 'Âº': 'º', 'Âª': 'ª', 'Â ': ' ', 'Â´': '´', 'Â¨': '¨',
        'Â©': '©', 'Ã˜': 'Ø', 'Ã‘': 'Ñ', 'Ã‰': 'É', 'Ã§': 'ç', 'Ã¤': 'ä', 'Ã¸': 'ø'
    }
for old, new in replacements.items():
    text = text.replace(old, new)
path.write_text(text, encoding='utf-8')
print('Replaced mojibake sequences in config.js.')